# Why the pixel sum never converged — slide figures

Standalone figures for the group talk of 2026-10-20. Last time this group saw
the diffuse-sky Faraday signal computed as a sum over HEALPix pixels; the
2026-08-18 audit (tag `audit-2026-08-18`) showed that sum does not converge.
This notebook tells that story in plots.

It depends only on numpy/healpy/h5py/astropy/matplotlib and the input maps in
`data/` — nothing from `src/` or `scripts/`. PNGs are written to
`notebooks/slides_pixel_sum/`, which is git-ignored; rerun the notebook to
regenerate them.

In [ ]:
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

REPO = Path.cwd().resolve()
while not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
DATA = REPO / "data"
OUT = REPO / "notebooks" / "slides_pixel_sum"
OUT.mkdir(exist_ok=True)

C = 299792458.0  # m/s


def lam2(freq_mhz):
    """Wavelength squared in m^2."""
    return (C / (np.asarray(freq_mhz) * 1e6)) ** 2


# Slides are 16:9; draw at the size the figure is shown so fonts read.
plt.rcParams.update(
    {
        "figure.figsize": (8.0, 4.5),
        "figure.dpi": 110,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.size": 14,
        "axes.titlesize": 15,
        "axes.labelsize": 14,
        "legend.fontsize": 12,
    }
)


def save(fig, name):
    path = OUT / f"{name}.png"
    fig.savefig(path)
    print("wrote", path.relative_to(REPO))

## Inputs

- `faraday2020v2.hdf5` — Hutschenreuter et al. (2022) Faraday sky, RING,
  nside 512, rad m$^{-2}$ (posterior mean and std).
- `wmap_band_iqumap_r9_9yr_K_v5.fits` — WMAP 9-yr K band, NESTED → RING,
  mK. Q/U are healpy/COSMO.
- `haslam408_dsds_Remazeilles2014.fits` — 408 MHz, RING, K.

In [ ]:
with h5py.File(DATA / "faraday2020v2.hdf5", "r") as f:
    RM = f["faraday_sky_mean"][:]
    RM_STD = f["faraday_sky_std"][:]

# WMAP K band: mK thermodynamic -> K Rayleigh-Jeans at 23 GHz
T_CMB = 2.7255
x = 6.62607015e-34 * 23e9 / (1.380649e-23 * T_CMB)
TH_TO_RJ = x**2 * np.exp(x) / np.expm1(x) ** 2
with fits.open(DATA / "wmap_band_iqumap_r9_9yr_K_v5.fits") as h:
    d = h["Stokes Maps"].data
    Q23 = hp.reorder(d["Q_POLARISATION"].astype(np.float64), n2r=True)
    U23 = hp.reorder(d["U_POLARISATION"].astype(np.float64), n2r=True)
Q23 *= 1e-3 * TH_TO_RJ
U23 *= 1e-3 * TH_TO_RJ

BETA_QU = -2.8  # config.BETA_QU


def sky_qu(freq_mhz):
    """WMAP K Q, U power-law scaled to freq_mhz, K_RJ, healpy/COSMO."""
    s = (freq_mhz / 23e3) ** BETA_QU
    return Q23 * s, U23 * s


NSIDE = hp.npix2nside(RM.size)
assert hp.npix2nside(Q23.size) == NSIDE  # common native grid, never degraded
print("nside", NSIDE, "| RM rms", RM.std(), "rad/m^2")

## 1. One source at one Faraday depth: the rotation is spectrally resolved

The Faraday phase of $P = Q + iU$ is $2\,\mathrm{RM}\,\lambda^2$. A channel of
width $\Delta\nu$ spans $\delta\lambda^2 = 2\lambda^2\,\Delta\nu/\nu$, so the
phase advances by $2\,\mathrm{RM}\,\delta\lambda^2$ per channel. The
rotation is sampled without aliasing while that step is below $\pi$, i.e.
for $|\mathrm{RM}| < \pi / (2\,\delta\lambda^2)$.

Channels: parent 25 kHz; zoom 25 kHz / 64 = 390.625 Hz
(`channelization.ZOOM_STEP_HZ`).

In [ ]:
PARENT_HZ = 25e3
ZOOM_HZ = PARENT_HZ / 64
RM_SRC = 100.0  # rad/m^2
BANDS_MHZ = (10.0, 30.0, 50.0)

print(f"{'nu':>4} {'chan':>6} {'dlam2 [m^2]':>12} "
      f"{'rad/chan @RM=100':>17} {'RM_max':>8}")
for f0 in BANDS_MHZ:
    for name, dnu in (("parent", PARENT_HZ), ("zoom", ZOOM_HZ)):
        dl2 = 2 * lam2(f0) * dnu / (f0 * 1e6)
        print(f"{f0:4.0f} {name:>6} {dl2:12.3e} "
              f"{2 * RM_SRC * dl2:17.3f} {np.pi / (2 * dl2):8.0f}")
    period = np.pi * f0 * 1e6 / (2 * RM_SRC * lam2(f0))
    print(f"     one turn of P at RM=100: {period:.0f} Hz")

In [ ]:
def channel_average(f0_mhz, rm, width_hz, n_chan, n_sub=64):
    """Top-hat channel means of exp(2i rm lam^2), centred on f0.

    Returns channel centres (Hz offset from f0) and the complex means.
    The exact lam^2(nu) is sampled at n_sub points per channel.
    """
    k = np.arange(n_chan) - n_chan // 2
    sub = (np.arange(n_sub) + 0.5) / n_sub - 0.5
    off = (k[:, None] + sub[None, :]) * width_hz
    p = np.exp(2j * rm * lam2(f0_mhz + off * 1e-6))
    return k * width_hz, p.mean(axis=1)


ZOOM_BANDS_MHZ = (30.0, 50.0)
C_Q, C_U = "#0072B2", "#D55E00"

# Q and U on the same axes, one parent channel (64 zoom bins) wide
fig, axes = plt.subplots(
    1, 2, figsize=(10.0, 3.8), sharey=True, constrained_layout=True
)
for ax, f0 in zip(axes, ZOOM_BANDS_MHZ):
    x_fine, p_fine = channel_average(f0, RM_SRC, ZOOM_HZ / 32, 64 * 32,
                                     n_sub=1)
    x_zoom, p_zoom = channel_average(f0, RM_SRC, ZOOM_HZ, 64)
    for part, col, lab in ((np.real, C_Q, "Q"), (np.imag, C_U, "U")):
        ax.plot(x_fine / 1e3, part(p_fine), color=col, lw=0.8, alpha=0.45)
        ax.plot(x_zoom / 1e3, part(p_zoom), ".", color=col, ms=6,
                label=lab)
    ax.axhline(0, color="k", lw=0.4)
    ax.set_ylim(-1.15, 1.15)
    ax.set_xlim(-PARENT_HZ / 2e3, PARENT_HZ / 2e3)
    dl2 = 2 * lam2(f0) * ZOOM_HZ / (f0 * 1e6)
    ax.set_title(f"{f0:.0f} MHz:  {2 * RM_SRC * dl2:.2f} rad / zoom bin")
    ax.set_xlabel(f"$\\nu - {f0:.0f}$ MHz  [kHz]")
axes[0].set_ylabel("fraction of $|P_0|$")
fig.legend(*axes[1].get_legend_handles_labels(),
           loc="outside right center")
save(fig, "01_single_source_rm100")

**Figure 1.** Fractional $Q$ and $U$ of one linearly polarized source,
$\mathrm{RM} = 100$ rad m$^{-2}$, seen by an ideal calibrated zenith
polarimeter (no beam, no leakage), across one 25 kHz parent channel at 30
and 50 MHz. Dots: means over each of the 64 zoom bins (ideal 390.6 Hz
top-hats, not the PFB response); lines: the unbinned rotation
$P_0\,e^{2i\,\mathrm{RM}\,\lambda^2}$. The zoom bins follow the rotation
(0.52 and 0.11 rad per bin; five turns and about one across the parent
channel), and $U$ lagging $Q$ by a quarter turn shows the angle rotating.
The parent channel alone would average $P$ to near zero. Below ~14 MHz the
zoom bins no longer resolve RM = 100 (see the table above).

## 2. The previous approach: rotate every pixel, then sum

The diffuse sky was rotated pixel by pixel,
$(Q + iU)_\mathrm{rot}(\hat n) = (Q + iU)(\hat n)\,
e^{2i\,\mathrm{RM}(\hat n)\,\lambda^2}$, at native nside 512, and then
summed against the beam.

In [ ]:
from matplotlib.colors import Normalize, SymLogNorm

F_MAP = 30.0
KK = 1e3  # plot unit, K
NORMS = {
    "linear": Normalize(vmin=-20.0, vmax=20.0),
    # the scale of the earlier figure (paper_plots.ipynb, cell 20)
    "symlog": SymLogNorm(linthresh=1.0, linscale=1, vmin=-600.0,
                         vmax=600.0, base=10),
}


def qu_rot_maps(freq_mhz, nside=NSIDE):
    """Intrinsic and rotated (Q, U) at freq_mhz.

    nside < native reproduces the earlier figure: Q, U and RM are
    ud_graded first and the rotation applied on the coarse grid.
    """
    q, u = sky_qu(freq_mhz)
    rm = RM
    if nside != NSIDE:
        q, u, rm = (hp.ud_grade(m, nside) for m in (q, u, rm))
    p_rot = (q + 1j * u) * np.exp(2j * rm * lam2(freq_mhz))
    return q, u, p_rot.real, p_rot.imag


def moll_image(m, xsize=1000):
    """Galactic Mollweide raster of a RING map (longitude increasing left)."""
    proj = hp.projector.MollweideProj(xsize=xsize, flipconv="astro")
    ns = hp.npix2nside(m.size)
    img = proj.projmap(m, lambda x, y, z: hp.vec2pix(ns, x, y, z))
    return np.ma.masked_invalid(img), proj.get_extent()


def plot_qu_maps(nside=NSIDE, scale="linear"):
    q, u, qr, ur = qu_rot_maps(F_MAP, nside)
    norm = NORMS[scale]
    panels = {
        (0, 0): (q, "$Q$"),
        (1, 0): (u, "$U$"),
        (0, 1): (qr, f"$Q$ rotated, {F_MAP:.0f} MHz"),
        (1, 1): (ur, f"$U$ rotated, {F_MAP:.0f} MHz"),
    }
    # Columns sit GAP_IN apart, leaving room for arrows drawn on the slide.
    W_IN, H_IN, MAP_W_IN, GAP_IN = 12.0, 6.4, 4.6, 2.0
    MAP_H_IN = MAP_W_IN / 2
    fig = plt.figure(figsize=(W_IN, H_IN))
    x0 = (W_IN - 2 * MAP_W_IN - GAP_IN) / 2
    t = np.linspace(0, 2 * np.pi, 400)
    for (r, c), (m, title) in panels.items():
        left = x0 + c * (MAP_W_IN + GAP_IN)
        bottom = H_IN - 0.45 - (r + 1) * MAP_H_IN - r * 0.45
        ax = fig.add_axes([left / W_IN, bottom / H_IN,
                           MAP_W_IN / W_IN, MAP_H_IN / H_IN])
        img, ext = moll_image(m / KK)
        # ~one raster pixel per printed pixel, nearest: no colour blending
        ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r",
                  norm=norm, interpolation="nearest")
        ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
        ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
        ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
        ax.set_axis_off()
        ax.set_title(title)
    cax = fig.add_axes([0.3, 0.2 / H_IN, 0.4, 0.18 / H_IN])
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="RdBu_r"),
                      cax=cax, orientation="horizontal")
    cb.set_label(f"brightness temperature at {F_MAP:.0f} MHz  [$10^3$ K]")
    return fig


save(plot_qu_maps(), "02_qu_maps_rotated")

**Figure 2.** Galactic Mollweide maps of the diffuse polarized sky at
30 MHz: WMAP 9-yr K-band $Q$, $U$ (K$_\mathrm{RJ}$, healpy/COSMO) scaled with
$\beta = -2.8$, before (left) and after (right) multiplying each native
nside-512 pixel by $e^{2i\,\mathrm{RM}\,\lambda^2}$ with the Hutschenreuter et
al. (2022) posterior-mean RM. Linear colour scale, $\pm 2\times10^4$ K,
shared by all panels. The intrinsic maps show the large-scale structure
(Galactic plane, North Polar Spur) under WMAP's per-pixel noise, which
dominates at high latitude. After rotation the large-scale structure is gone:
the phase changes by ~1700 rad between neighbouring pixels, so neighbouring
pixels have unrelated signs. Only $|P|$ survives, which is why the plane is
still visible as a band of larger scatter. The projection resamples the
nside-512 maps, so the exact speckle pattern is a display artefact; the loss
of structure is not.

## 3. The bug: the pixel grid aliases the Faraday phase

The phase applied to each pixel is $2\lambda^2\,\mathrm{RM}$: the RM map
times 1800 rad (10 MHz), 200 rad (30 MHz) or 72 rad (50 MHz) per rad m$^{-2}$.
The same Nyquist test as Fig. 1 applies along the sky: the phase must change
by less than $\pi$ between samples, here between neighbouring pixels.

In [ ]:
# Phase step between neighbouring pixels, over the whole sky
nbr = hp.get_all_neighbours(NSIDE, np.arange(RM.size))
d_rm = np.where(nbr >= 0, np.abs(RM[nbr] - RM[None]), np.nan)
step_rm = np.nanmean(d_rm, axis=0)  # mean |dRM| to the 8 neighbours
print("mean phase step to neighbouring pixels [rad]")
print(f"{'MHz':>6} {'p10':>7} {'p50':>7} {'p90':>7}")
for f0 in (1400.0, 408.0, 50.0, 30.0, 10.0):
    p = 2 * lam2(f0) * np.percentile(step_rm, (10, 50, 90))
    print(f"{f0:6.0f} {p[0]:7.2f} {p[1]:7.2f} {p[2]:7.1f}")
del nbr, d_rm

In [ ]:
RM_NORM = Normalize(vmin=-150, vmax=150)  # shared with Fig. 4b

fig = plt.figure(figsize=(8.0, 4.6))
ax = fig.add_axes([0.02, 0.2, 0.96, 0.75])
img, ext = moll_image(RM)
ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r", norm=RM_NORM,
          interpolation="nearest")
t = np.linspace(0, 2 * np.pi, 400)
ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
ax.set_axis_off()
cax = fig.add_axes([0.3, 0.1, 0.4, 0.035])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=RM_NORM, cmap="RdBu_r"),
                  cax=cax, orientation="horizontal")
cb.set_label("RM  [rad m$^{-2}$]")
save(fig, "03_rm_map")

**Figure 3.** Hutschenreuter et al. (2022) posterior-mean RM, Galactic
Mollweide, native nside 512, linear scale clipped at $\pm150$ rad m$^{-2}$.
The phase applied per pixel is this map times $2\lambda^2$: ×1800 at 10 MHz,
×200 at 30 MHz, ×72 at 50 MHz.

In [ ]:
# (a) 1-D cut: the pixel samples of the phase along a 2-degree line
CUT_L, CUT_B, CUT_LEN = 30.0, 0.0, 2.0  # deg; vertical, across the plane
CUT_FREQS = (1400.0, 408.0, 50.0, 30.0)
n_fine = 400_000
s = np.linspace(-CUT_LEN / 2, CUT_LEN / 2, n_fine)  # deg along the cut
pix = hp.ang2pix(NSIDE, CUT_L + 0 * s, CUT_B + s, lonlat=True)
edges = np.flatnonzero(np.r_[True, pix[1:] != pix[:-1], True])
s_pix = np.array([s[a:b].mean() for a, b in zip(edges[:-1], edges[1:])])
rm_pix = RM[pix[edges[:-1]]]
rm_line = np.interp(s, s_pix, rm_pix)  # smooth stand-in between centres
print(len(s_pix), "pixels along the cut;",
      f"RM {rm_pix.min():.0f} .. {rm_pix.max():.0f} rad/m^2")

fig, axes = plt.subplots(
    len(CUT_FREQS) + 1, 1, figsize=(10.0, 8.0), sharex=True,
    constrained_layout=True,
)
x_arc, xp_arc = s * 60, s_pix * 60
axes[0].plot(x_arc, rm_line, color="0.4", lw=1)
axes[0].plot(xp_arc, rm_pix, "o", color="k", ms=4)
axes[0].set_ylabel("RM\n[rad m$^{-2}$]")
for ax, f0 in zip(axes[1:], CUT_FREQS):
    ax.plot(x_arc, np.cos(2 * lam2(f0) * rm_line), color="0.55",
            lw=0.4)
    ax.plot(xp_arc, np.cos(2 * lam2(f0) * rm_pix), "o", color=C_Q, ms=4)
    step = 2 * lam2(f0) * np.median(np.abs(np.diff(rm_pix)))
    lab = f"{f0/1e3:.1f} GHz" if f0 >= 1e3 else f"{f0:.0f} MHz"
    ax.set_ylabel(f"$Q/|P_0|$\n{lab}")
    step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
    ax.set_title(f"{lab}:  {step_txt} rad per pixel", fontsize=12,
                 loc="right", pad=2)
    ax.set_ylim(-1.2, 1.2)
axes[-1].set_xlabel(
    f"position along the cut, $b$ at $l = {CUT_L:.0f}^\\circ$  [arcmin]"
)
save(fig, "04a_cut_phase_sampling")

**Figure 4a.** Faraday rotation sampled along a $2^\circ$ cut across the
Galactic plane at $l = 30^\circ$. Top: RM at the centres of the
nside-512 pixels the cut crosses (dots, one value per pixel, as the
simulation uses them) and linearly interpolated between them (line, a
stand-in for the smooth sky). Below: $Q/|P_0| = \cos(2\lambda^2\,\mathrm{RM})$
for a source of intrinsic angle 0, at the pixels (dots) and along the line.
Titles give the median phase step between consecutive pixels. At 1.4 GHz the
pixels follow the rotation; at 408 MHz the rotation turns more than once
between pixels and the dots no longer trace it; at 50 and 30 MHz each pixel
holds tens to hundreds of turns, so the pixel values are effectively random
phases. This cut lies in the plane, where gradients are largest; over the
whole sky the median step at 30 MHz is ~130 rad and the 10th percentile ~31
rad, all far above $\pi$.

In [ ]:
# (b) 5-degree patch of the wrapped phase, pixels drawn as they are.
# Off the plane, with |RM| < 150 so the RM panel shares Fig. 3's scale.
PATCH_L, PATCH_B, PATCH_DEG = 250.0, -15.0, 5.0
PATCH_FREQS = (1400.0, 408.0, 50.0, 30.0)
RESO = 0.5  # arcmin per raster pixel; an nside-512 pixel is ~6.9'
gproj = hp.projector.GnomonicProj(
    rot=(PATCH_L, PATCH_B), xsize=int(PATCH_DEG * 60 / RESO), reso=RESO
)


def patch(m):
    return gproj.projmap(m, lambda x, y, z: hp.vec2pix(NSIDE, x, y, z))


in_patch = hp.query_disc(
    NSIDE, hp.ang2vec(PATCH_L, PATCH_B, lonlat=True),
    np.radians(PATCH_DEG / 2),
)
print(f"RM in patch: {RM[in_patch].min():.0f} .. {RM[in_patch].max():.0f}; "
      f"median step {np.median(step_rm[in_patch]):.2f} rad/m^2 "
      f"(sky median {np.median(step_rm):.2f})")

gext = gproj.get_extent()
fig, axes = plt.subplots(1, 5, figsize=(13.0, 3.6),
                         constrained_layout=True)
axes[0].imshow(patch(RM), origin="lower", extent=gext, cmap="RdBu_r",
               norm=RM_NORM, interpolation="nearest")
axes[0].set_title("RM")
fig.colorbar(plt.cm.ScalarMappable(norm=RM_NORM, cmap="RdBu_r"),
             ax=axes[0], orientation="horizontal", fraction=0.06, pad=0.04,
             label="RM  [rad m$^{-2}$]")
for ax, f0 in zip(axes[1:], PATCH_FREQS):
    ph = np.angle(np.exp(2j * lam2(f0) * patch(RM)))
    im = ax.imshow(ph, origin="lower", extent=gext, cmap="twilight",
                   vmin=-np.pi, vmax=np.pi, interpolation="nearest")
    lab = f"{f0/1e3:.1f} GHz" if f0 >= 1e3 else f"{f0:.0f} MHz"
    step = 2 * lam2(f0) * np.median(step_rm[in_patch])
    step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
    ax.set_title(f"{lab}: {step_txt} rad/pixel", fontsize=13)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
cb = fig.colorbar(im, ax=axes[1:], orientation="horizontal", fraction=0.06,
                  pad=0.04, aspect=60)
cb.set_ticks([-np.pi, 0, np.pi], labels=["$-\\pi$", "0", "$\\pi$"])
cb.set_label("phase $2\\lambda^2\\,\\mathrm{RM}$, wrapped to $[-\\pi, \\pi)$")
save(fig, "04b_patch_phase_wrapped")

**Figure 4b.** A $5^\circ \times 5^\circ$ gnomonic patch centred on
$(l, b) = (250^\circ, -15^\circ)$, off the Galactic plane, drawn at 0.5' per
screen pixel so each nside-512 pixel (~6.9') shows as a tile. Left: RM, on
the colour scale of Fig. 3. Right: the phase $2\lambda^2\,\mathrm{RM}$ wrapped
to $[-\pi,\pi)$ (cyclic colour map); titles give the median phase step
between neighbouring pixels in the patch. The patch's median RM step
(1.65 rad m$^{-2}$) is above the sky median (0.65), but the sky median at
30 MHz is still 130 rad per pixel. At 1.4 GHz and 408 MHz the phase forms
smooth fringes the pixels resolve; at 50 and 30 MHz neighbouring tiles have
unrelated phases.

### 4c. Zoom into a few pixels: what a pixel stores vs what it should hold

A map pixel should hold the sky *averaged over its area*; the pixel sum uses
the rotated polarization at the pixel's *one* RM. To see the sky inside a
pixel we need RM between pixel centres: we interpolate it linearly
(`hp.get_interp_val`) and draw it at 1.6″, the pixel size of nside 131072.
The pixel average is the mean over the $4^8 = 65536$ nside-131072 pixels
inside each nside-512 pixel. The spot is chosen where the RM step between
pixels is close to the sky median (0.65 rad m$^{-2}$), so it is typical.

In [ ]:
ZOOM_L, ZOOM_B = 250.049, -16.179  # inside the Fig. 4b patch
ZOOM_ARCMIN, ZOOM_NX = 15.0, 600
Z_FREQS = (408.0, 30.0)


def pixel_average_phasor(pix_ring, k_sub, freqs):
    """<exp(2i lam^2 RM)> over each pixel, RM bilinear on 4**k sub-pixels."""
    ns_sub, nsub = NSIDE * 2**k_sub, 4**k_sub
    nest = hp.ring2nest(NSIDE, pix_ring)
    out = {f: np.empty(pix_ring.size, complex) for f in freqs}
    for idx in np.array_split(np.arange(pix_ring.size),
                              max(1, pix_ring.size * nsub // 2_000_000)):
        kids = (nest[idx, None] * nsub + np.arange(nsub)).ravel()
        th, ph = hp.pix2ang(ns_sub, kids, nest=True)
        rm_sub = hp.get_interp_val(RM, th, ph).reshape(idx.size, nsub)
        for f in freqs:
            out[f][idx] = np.exp(2j * lam2(f) * rm_sub).mean(axis=1)
    return out


zproj = hp.projector.GnomonicProj(rot=(ZOOM_L, ZOOM_B), xsize=ZOOM_NX,
                                  reso=ZOOM_ARCMIN / ZOOM_NX)
zext = zproj.get_extent()
zx, zy = zproj.ij2xy()
zvec = [v.ravel() for v in zproj.xy2vec(zx, zy)]
zth, zph = hp.vec2ang(np.stack(zvec, axis=1))
rm_fine = hp.get_interp_val(RM, zth, zph).reshape(zx.shape)
zpix = hp.vec2pix(NSIDE, *zvec).reshape(zx.shape)
zup, zinv = np.unique(zpix, return_inverse=True)
zinv = zinv.reshape(zx.shape)

# the pixel average must itself be converged in the sub-sampling
for k in (6, 7, 8):
    a = pixel_average_phasor(zup, k, (30.0,))[30.0]
    print(f"4**{k} sub-pixels: rms |pixel average| at 30 MHz = "
          f"{np.sqrt(np.mean(np.abs(a)**2)):.4f}")
zavg = pixel_average_phasor(zup, 8, Z_FREQS)
print("RM step to neighbours here:",
      np.round(step_rm[zup], 2), "rad/m^2")

def plot_zoom(outlines=True):
    fig, axes = plt.subplots(2, 3, figsize=(11.5, 8.2),
                             constrained_layout=True)
    q_norm = Normalize(-1, 1)
    cols = ("sky at 1.6″", "averaged over each pixel",
            "one value per pixel\n(the pixel sum)")
    for r, f0 in enumerate(Z_FREQS):
        maps = (
            np.cos(2 * lam2(f0) * rm_fine),
            zavg[f0].real[zinv],
            np.cos(2 * lam2(f0) * RM[zpix]),
        )
        for c, m in enumerate(maps):
            ax = axes[r, c]
            ax.imshow(m, origin="lower", extent=zext, cmap="RdBu_r",
                      norm=q_norm, interpolation="nearest")
            if outlines:
                for p in zup:
                    b = hp.boundaries(NSIDE, p, step=4)
                    bx, by = zproj.vec2xy(*b)
                    ax.plot(np.r_[bx, bx[:1]], np.r_[by, by[:1]],
                            color="k", lw=0.8)
            ax.set_xlim(zext[:2])
            ax.set_ylim(zext[2:])
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(cols[c])
        step = 2 * lam2(f0) * np.median(step_rm[zup])
        step_txt = f"{step:,.0f}" if step >= 10 else f"{step:.2g}"
        axes[r, 0].set_ylabel(f"{f0:.0f} MHz\n{step_txt} rad per pixel")
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=q_norm, cmap="RdBu_r"),
                      ax=axes, orientation="horizontal", fraction=0.05,
                      pad=0.03, aspect=50)
    cb.set_label("rotated $Q$ / unrotated $Q$")
    return fig


save(plot_zoom(), "04c_pixel_zoom")

**Figure 4c.** A 15′ zoom at $(l, b) = (250.05^\circ, -16.18^\circ)$, inside
the Fig. 4b patch, where the RM step between neighbouring pixels is close to
the sky median. Black diamonds are nside-512 pixels (~6.9′). The sky is uniformly polarized in $Q$ ($U = 0$) to
isolate the Faraday factor, so the colour is rotated over unrotated $Q$.
Left: the rotated $Q$ drawn at 1.6″ (nside-131072 resolution), with RM
interpolated linearly between pixel centres; the kinks in the fringes are
where the interpolation switches neighbours. Middle: that field averaged over
each nside-512 pixel, which is what a map pixel should hold. Right: the
rotated $Q$ at each pixel's single RM, which is what the pixel sum used. At
408 MHz (~0.7 rad per pixel) all three agree. At 30 MHz (~130 rad, ~20 turns
per pixel) the fringes average to nearly zero inside every pixel, but the
pixel sum keeps one point from each at full amplitude with an arbitrary sign.
Physical depolarization makes the map *fainter*; the pixel sum makes it
*scrambled at full brightness*. The interpolated sky is itself a model: the
real sky inside a pixel has RM structure the map does not contain.

### 4d. The whole sky: Fig. 2 with the pixel average

Under linear interpolation the RM inside a pixel is a linear ramp, and the
pixel average of $e^{2i\lambda^2\mathrm{RM}}$ is analytic: the phase at
the pixel centre times $\mathrm{sinc}(g/2)$, with $g$ the phase change
across the pixel (from the rms RM step to the four edge neighbours). That
is exact for the model, needs no sub-pixel grid, and holds in the Galactic
plane, where $g$ is thousands of radians and the factor is $\sim 10^{-4}$.
The brute-force 4c average on $4^7 = 16384$ sub-pixels per pixel (nside
65536, about 20 min on 8 cores, cached under `generated_data/cache/`) is
kept below only as a check of the analytic factor.


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

K_FULL = 7
CACHE = REPO / "generated_data" / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
avg_path = CACHE / f"slides_pixavg_phasor_k{K_FULL}_{F_MAP:.0f}MHz.npy"


def _avg_block(lohi):
    lo, hi = lohi
    return pixel_average_phasor(np.arange(lo, hi), K_FULL, (F_MAP,))[F_MAP]


if avg_path.exists():
    phasor_avg = np.load(avg_path)
else:
    bounds = np.linspace(0, RM.size, 129).astype(int)
    with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as ex:
        parts = list(ex.map(_avg_block, zip(bounds[:-1], bounds[1:])))
    phasor_avg = np.concatenate(parts)
    np.save(avg_path, phasor_avg)

# Analytic pixel-average factor under linear interpolation (used by 4d and
# section 6): sinc(g/2) with g the phase change across the pixel.
nbr_all = hp.get_all_neighbours(NSIDE, np.arange(RM.size))  # SW W NW N NE E SE S
edge = nbr_all[[1, 3, 5, 7]]
ok = edge >= 0
d_rm = np.where(ok, RM[np.where(ok, edge, 0)] - RM[None, :], np.nan)
rm_step = np.sqrt(np.nanmean(d_rm**2, axis=0))  # rad/m^2 per pixel
PIX_RAD = np.sqrt(4 * np.pi / RM.size)


def linear_factor(freq_mhz, signed=False):
    g = 2 * lam2(freq_mhz) * rm_step
    f = np.sinc(g / (2 * np.pi))  # np.sinc(x) = sin(pi x)/(pi x)
    return f if signed else np.abs(f)


fac30 = linear_factor(F_MAP)
brute = np.abs(phasor_avg)
hi = np.abs(hp.pix2ang(NSIDE, np.arange(RM.size), lonlat=True)[1]) > 30
print(f"pixel-average amplitude at {F_MAP:.0f} MHz, |b| > 30: sky rms "
      f"brute force {np.sqrt(np.mean(brute[hi]**2)):.4f}, analytic "
      f"{np.sqrt(np.mean(fac30[hi]**2)):.4f}; medians "
      f"{np.median(brute[hi]):.4f} / {np.median(fac30[hi]):.4f}")

In [ ]:
q, u, qr, ur = qu_rot_maps(F_MAP)
p_avg = (q + 1j * u) * np.exp(2j * lam2(F_MAP) * RM) * linear_factor(F_MAP, signed=True)
panels = {
    (0, 0): (q, f"$Q$ at {F_MAP:.0f} MHz"),
    (0, 1): (qr, "nearest neighbour (the old pixel sum):\n"
                 f"$Q$ rotated with one RM per nside-{NSIDE} pixel"),
    (0, 2): (p_avg.real, "linear interpolation:\n"
                         "$Q$ rotated, averaged inside each pixel"),
}
W_IN, H_IN, MAP_W_IN, GAP_IN = 13.0, 3.4, 4.0, 0.3
MAP_H_IN = MAP_W_IN / 2
fig = plt.figure(figsize=(W_IN, H_IN))
x0 = (W_IN - 3 * MAP_W_IN - 2 * GAP_IN) / 2
t = np.linspace(0, 2 * np.pi, 400)
norm = NORMS["linear"]
for (r, c), (m, title) in panels.items():
    ax = fig.add_axes([(x0 + c * (MAP_W_IN + GAP_IN)) / W_IN,
                       (H_IN - 0.75 - MAP_H_IN) / H_IN,
                       MAP_W_IN / W_IN, MAP_H_IN / H_IN])
    # the pixel-averaged map is ~30x fainter: give it its own scale
    own = c == 2
    if own:
        vown = np.percentile(np.abs(m), 99) / KK
        pnorm = Normalize(-vown, vown)
    img, ext = moll_image(m / KK)
    ax.imshow(img, origin="lower", extent=ext, cmap="RdBu_r",
              norm=pnorm if own else norm, interpolation="nearest")
    ax.plot(ext[1] * np.cos(t), ext[3] * np.sin(t), color="k", lw=0.8)
    ax.set_xlim(ext[0] * 1.01, ext[1] * 1.01)
    ax.set_ylim(ext[2] * 1.02, ext[3] * 1.02)
    ax.set_axis_off()
    ax.set_title(title, fontsize=13)
    print(f"{title.replace(chr(10), ' '):55s} rms {m.std()/KK:6.2f}e3 K")
# shared scale under the first two maps, own scale under the third
cax = fig.add_axes([(x0 + 0.6) / W_IN, 0.2 / H_IN,
                    (2 * MAP_W_IN + GAP_IN - 1.2) / W_IN, 0.15 / H_IN])
cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap="RdBu_r"),
                  cax=cax, orientation="horizontal")
cb.set_label("brightness temperature  [$10^3$ K]")
cax2 = fig.add_axes([(x0 + 2 * (MAP_W_IN + GAP_IN) + 0.6) / W_IN,
                     0.2 / H_IN, (MAP_W_IN - 1.2) / W_IN, 0.15 / H_IN])
cb2 = fig.colorbar(plt.cm.ScalarMappable(norm=pnorm, cmap="RdBu_r"),
                   cax=cax2, orientation="horizontal")
cb2.set_label("brightness temperature  [$10^3$ K]")
save(fig, "04d_fullsky_point_vs_average")

**Figure 4d.** WMAP K $Q$ at 30 MHz (as Fig. 2), native nside 512. Left:
intrinsic. Middle: nearest neighbour, the old pixel sum: rotated with each
pixel's single RM (Fig. 2). Right: linear interpolation: rotated with the
pixel-centre RM and multiplied by the analytic pixel-average factor
$\mathrm{sinc}(g/2)$, with $g$ the phase change across the pixel; the
intrinsic $Q, U$ are held at their pixel values. Left and middle share
Fig. 2's linear scale; the right panel has its own, clipped at the 99th
percentile of $|Q|$, because on the shared scale it is blank. Averaging
inside the pixel removes nearly all of the rotated polarization; the pixel
sum kept it at full brightness with scrambled signs. The factor is exact for
the linear model everywhere, Galactic plane included, so no sub-pixel grid
is needed; the brute-force 16384-sub-pixel average agrees with it in sky rms
to about 40% at $|b| > 30^\circ$ (printed above), the difference being the
healpy interpolation's kinks and the square-pixel sinc. The interpolated RM
is itself a model: the real sky inside a pixel has RM structure the map does
not contain (section 6).


### 4e. Why a wide beam suddenly cares about small scales

A wide beam is a low-pass filter: it reads only the low-$\ell$ content of the
sky it looks at. Faraday rotation moves polarized power from low $\ell$ to
the fringe scale. The angular power spectra of the three maps of Fig. 4d
show where the power sits, and what is left where the beam reads.

In [ ]:
ELL_BEAM = 30  # production arm's beam lmax (docs/measurement-model.md)
zeros = np.zeros_like(q)
spectra = {
    "intrinsic": (q, u),
    "rotated, one value per pixel": (qr, ur),
    "rotated, averaged over each pixel": (p_avg.real, p_avg.imag),
}
cl = {}
for name, (mq, mu) in spectra.items():
    c = hp.anafast([zeros, mq, mu], pol=True)
    cl[name] = c[1] + c[2]  # EE + BB, K^2
ell = np.arange(cl["intrinsic"].size)

# random signs from pixel to pixel: white, C_l = <|P|^2> * pixel area
floor = np.mean(qr**2 + ur**2) * hp.nside2pixarea(NSIDE)

fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
ax.axvspan(2, ELL_BEAM, color="0.9", zorder=0)
ax.text(np.sqrt(2 * ELL_BEAM), 0.5, "LuSEE beam\nreads $\\ell \\lesssim 30$",
        transform=ax.get_xaxis_transform(), ha="center", color="0.35",
        fontsize=12)
for (name, c), col in zip(cl.items(), ("0.2", C_U, C_Q)):
    ax.loglog(ell[2:], c[2:], color=col, lw=1.6, label=name)
ax.axhline(floor, color=C_U, ls="--", lw=1,
           label="random signs per pixel: $\\langle|P|^2\\rangle\\,\\Omega_\\mathrm{pix}$")
ax.set_xlim(2, ell[-1])
ax.set_xlabel("multipole $\\ell$")
ax.set_ylabel(f"$C_\\ell^{{EE}} + C_\\ell^{{BB}}$ at {F_MAP:.0f} MHz  [K$^2$]")
ax.legend(fontsize=11, loc="upper right")
for name, c in cl.items():
    print(f"{name:36s} mean C_l over 2<=l<=30: {c[2:ELL_BEAM+1].mean():.3e} K^2")
print(f"{'white floor':36s} {floor:.3e} K^2")
save(fig, "04e_cl_beam_window")

**Figure 4e.** Polarized angular power spectrum ($C_\ell^{EE} + C_\ell^{BB}$,
`hp.anafast`, nside 512) at 30 MHz of the three maps of Fig. 4d: WMAP K $Q, U$
scaled to 30 MHz (black), rotated with one RM per pixel as the pixel sum did
(orange), and rotated then averaged over each pixel (blue). Grey band: the
multipoles the LuSEE beam reads; the production arm truncates the beam at
$\ell = 30$ (raising it to 48 changes the I-only products by $8\times10^{-5}$;
the sharp horizon gives the polarized kernels a weak high-$\ell$ tail). The
intrinsic sky puts its power at low $\ell$, where the beam reads; its
high-$\ell$ flattening is WMAP's map noise. Rotation as the pixel sum did it
gives a flat spectrum at the white-noise level of pixels with random signs
(dashed): what the beam reads from it is set by the pixel count. Averaging
inside each pixel removes most of that power at every $\ell$, including where
the beam reads. The beam is still a low-pass filter; rotation emptied the
low-$\ell$ band, so what is left there is built from structure below the
pixel scale.

### 4f. The ensemble model's intrinsic spectrum vs WMAP K

The depth-depolarization branch draws the intrinsic polarized sky as a
Gaussian spin-2 field with $C_\ell \propto \exp[-\ell(\ell+1)/\ell_s^2]$,
$\ell_s = 8$ (`scripts/faraday_covariance.py:intrinsic_spectra`). Compare its
shape with WMAP K (the black curve of Fig. 4e), after removing WMAP's white
noise, with both normalized to the same power where the beam reads.

In [ ]:
cl_w = cl["intrinsic"]
noise_w = np.median(cl_w[1000:1500])  # flat high-ell floor: WMAP map noise
sig_w = cl_w - noise_w
fit = slice(2, 101)  # signal-dominated, inside WMAP K's 0.9 deg beam
slope, icpt = np.polyfit(np.log(ell[fit]), np.log(sig_w[fit]), 1)
print(f"WMAP K signal: C_l ~ l^{slope:.2f} over 2 <= l <= 100; "
      f"noise floor {noise_w:.0f} K^2")

band = slice(2, ELL_BEAM + 1)
w = (2 * ell + 1) / (4 * np.pi)


def model_cl(ell_s):
    c = np.exp(-ell * (ell + 1) / ell_s**2)
    c[:2] = 0
    return c


fig, ax = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
ax.axvspan(2, ELL_BEAM, color="0.9", zorder=0)
ref = np.sum(w[band] * sig_w[band])
ax.loglog(ell[2:300], sig_w[2:300], color="0.2", lw=1.6,
          label="WMAP K, noise removed")
ax.loglog(ell[2:300], np.exp(icpt) * ell[2:300] ** slope, color="0.2",
          ls=":", lw=1.2, label=f"power law, $\\ell^{{{slope:.1f}}}$")
for ell_s, col in ((4, "#9ec5fe"), (8, C_Q), (16, "#08306b")):
    c = model_cl(ell_s)
    c *= ref / np.sum(w[band] * c[band])
    ax.loglog(ell[2:300], c[2:300], color=col, lw=2 if ell_s == 8 else 1.2,
              label=f"model, $\\ell_s = {ell_s}$" + (" (used)" if ell_s == 8 else ""))
    frac = np.sum(w[band] * c[band]) / np.sum(w[2:] * c[2:])
    print(f"model l_s={ell_s:2d}: {100 * frac:5.1f}% of its power at l <= {ELL_BEAM}")
frac_w = np.sum(w[band] * sig_w[band]) / np.sum(w[2:101] * sig_w[2:101])
print(f"WMAP K signal: {100 * frac_w:5.1f}% of its 2 <= l <= 100 power at l <= {ELL_BEAM}")
ax.set_ylim(sig_w[2:300][sig_w[2:300] > 0].min() / 3, sig_w[2] * 5)
ax.set_xlim(2, 300)
ax.set_xlabel("multipole $\\ell$")
ax.set_ylabel(f"$C_\\ell^{{EE}} + C_\\ell^{{BB}}$ at {F_MAP:.0f} MHz  [K$^2$]")
ax.legend(fontsize=11, loc="lower left")
save(fig, "04f_intrinsic_spectrum_vs_wmap")

**Figure 4f.** Polarized angular power spectrum of WMAP 9-yr K $Q, U$ scaled
to 30 MHz ($\beta = -2.8$), with its white noise floor (median $C_\ell$ over
$1000 \le \ell \le 1500$) subtracted, and a power-law fit over
$2 \le \ell \le 100$ (above that WMAP's 0.9° beam suppresses the signal).
Blue: the ensemble model's intrinsic spectrum for $\ell_s = 4, 8, 16$
(8 is used), each scaled to WMAP's power over $\ell \le 30$ (grey), so only
shapes are compared. The model puts nearly all its power in the band the
beam reads; WMAP's spectrum is a power law with power at all $\ell$. In the
strong-rotation limit the speckle depends on the total intrinsic variance,
not its angular shape, so the shape matters mainly for the smooth component
and for how a total-RMS normalization splits between the two.

## 5. The pixel sum does not converge

Stored results of the 2026-08-18 audit (tag `audit-2026-08-18`): test 06,
`shuffle_control.py` (`audit/generated/shuffle_results.json`), and test 07,
`where_it_breaks.py` (values as plotted by `audit/scripts/make_fig.py`). Both
use a $\cos^2\theta$ lobe about a fixed Galactic axis, not the LuSEE beam,
at a single time; the diagnostic compares each measurement against the same
weights, so it does not depend on the beam shape.

In [ ]:
# test 06: band-averaged |P|^2 (total delay power, by Parseval), 30 MHz
NS_AUDIT = np.array([512, 1024, 2048])
POW = {
    ("actual RM map", "real"): [0.5671, 0.1242, 0.02522],
    ("actual RM map", "shuffled"): [0.9196, 0.1876, 0.03052],
    ("RM × 0.02", "real"): [1.3512, 1.0309, 0.9732],
    ("RM × 0.02", "shuffled"): [0.2774, 0.02446, 0.004481],
}
# test 07: |P|_Faraday / |P|_no Faraday vs lambda^2
SWEEP_L2 = np.array([0.01, 0.05, 0.14, 0.56, 2.25, 8.99, 35.95, 99.86,
                     898.76])
SWEEP = {
    512: [8.047e-1, 4.839e-1, 2.688e-1, 2.755e-2, 1.017e-2, 5.266e-3,
          3.145e-3, 8.448e-3, 3.953e-3],
    1024: [8.047e-1, 4.846e-1, 2.670e-1, 3.104e-2, 7.922e-3, 6.602e-3,
           2.612e-3, 2.432e-3, 3.188e-3],
    2048: [8.047e-1, 4.845e-1, 2.665e-1, 3.083e-2, 6.776e-3, 3.726e-3,
           3.672e-3, 2.379e-3, 1.298e-3],
}

fig, (axB, axC) = plt.subplots(1, 2, figsize=(12.0, 4.8),
                               constrained_layout=True)
for (name, kind), y in POW.items():
    col = C_Q if name.startswith("actual") else C_U
    real = kind == "real"
    axB.loglog(NS_AUDIT, y, "o-" if real else "s:", color=col,
               lw=2.2 if real else 1.4, ms=7 if real else 5,
               alpha=1 if real else 0.6, label=f"{name}, {kind}")
guide = POW[("actual RM map", "real")][0] * (NS_AUDIT / 512.0) ** -2
axB.loglog(NS_AUDIT, guide, color="0.6", lw=1, ls=(0, (1, 2)))
axB.text(1100, guide[1] * 2.2, "$\\propto 1/N_\\mathrm{pix}$", color="0.45")
axB.set_xticks(NS_AUDIT, [str(n) for n in NS_AUDIT])
axB.minorticks_off()
axB.set_xlabel("nside")
axB.set_ylabel("band-averaged $|P|^2$ at 30 MHz  [arb.]")
axB.legend(fontsize=10.5, loc="lower left")

axC.axvspan(lam2(50.0), lam2(10.0), color="0.92", zorder=0)
axC.text(np.sqrt(lam2(50.0) * lam2(10.0)), 0.35, "LuSEE\n50 → 10 MHz",
         ha="center", color="0.4", fontsize=11)
for n, mk, col in ((512, "^", "#9ec5fe"), (1024, "s", "#5b9bf8"),
                   (2048, "o", C_Q)):
    axC.loglog(SWEEP_L2, SWEEP[n], mk + "-", color=col, lw=1.8, ms=6,
               label=f"nside {n}")
axC.set_xlabel("$\\lambda^2$  [m$^2$]")
axC.set_ylabel("$|P|_\\mathrm{Faraday}\\,/\\,|P|_\\mathrm{no\\ Faraday}$")
top = axC.secondary_xaxis(
    "top", functions=(lambda l2: C / np.sqrt(l2) / 1e6,
                      lambda f: (C / (np.maximum(f, 1e-3) * 1e6)) ** 2))
top.set_xticks([3000, 1000, 300, 100, 30, 10],
               ["3000", "1000", "300", "100", "30", "10"])
top.set_xlabel("frequency  [MHz]")
axC.legend(fontsize=10.5, loc="lower left")
save(fig, "05_convergence")

**Figure 5.** Convergence of the beam-weighted pixel sum, from the
2026-08-18 audit ($\cos^2\theta$ beam about a fixed axis, one time step,
WMAP K $Q, U$ and the Hutschenreuter RM map synthesised to $\ell_\max = 1023$ so
that refining nside changes only the sampling). Left: band-averaged $|P|^2$
across the ±0.1 MHz band at 30 MHz against nside. With the real RM map it
falls 4× per nside doubling ($\propto 1/N_\mathrm{pix}$), the signature of a
random walk, and shuffling the RM values over the sky barely changes it. The
control, RM × 0.02, whose phase the grid resolves, levels off and is 200×
above its own shuffle at nside 2048: the method converges when the phase is
sampled. Right: beam-integrated $|P|$ with Faraday rotation over $|P|$ without,
against $\lambda^2$. The three resolutions agree down to $\lambda^2 \approx
1$ m$^2$ (~300 MHz) and separate below, throughout the LuSEE band (shaded).
A rigid rotation of the grid at nside 1024, which changes no physics, changes
$|P|$ at 30 MHz by 7.2×.

## 6. What to assume below the map scale

The pixel sum fails because the answer depends on RM structure the map does
not contain. Any prediction therefore assumes that structure. Three levels,
from the old approach upward:

1. **Nearest neighbour** — one RM per pixel, constant inside it. This *is*
   the pixel sum of §2–5: no depolarization inside the pixel, random sign
   between pixels.
2. **Linear interpolation** — RM varies linearly between pixel centres; the
   smoothest sky consistent with the map (§4c–4d used it). For a linear
   gradient the pixel average of $e^{2i\lambda^2\mathrm{RM}}$ is analytic,
   $|\mathrm{sinc}(g/2)|$ with $g$ the phase change across the pixel, so it
   needs no sub-pixel grid and is exact everywhere, Galactic plane included.
3. **Linear plus Gaussian sub-pixel structure** with structure function
   $D(\theta) = \langle(\Delta\mathrm{RM})^2\rangle = \Delta\phi_{1^\circ}^2\,(\theta/1^\circ)^\alpha$,
   anchored to the map at $1^\circ$. $\alpha$ is the small-scale slope of the
   RM structure function: 2 for a differentiable field (same as linear),
   5/3 Kolmogorov (Minter & Spangler 1996), 1 random-walk rough; the map
   itself gives 1.2 between $1^\circ$ and $8^\circ$ at $|b| > 60^\circ$.
   Gaussianity sets only the increment distribution; the scaling of the
   surviving power with $\lambda$ does not depend on it.

Level 3 is the ensemble model of `scripts/faraday_redistribution.py` (main)
and `configs/paper/covariance-column.json` (the column screen through the
real instrument).


In [ ]:
# 6a. Per-pixel depolarization factor under linear interpolation at three
# frequencies (linear_factor and rm_step are defined with Fig. 4d).
fig = plt.figure(figsize=(13, 6.2))
fig.suptitle(
    r"Linear RM interpolation: $|\langle e^{2i\lambda^2 RM}\rangle_{pix}|$ (top, log) "
    "and WMAP K Q rotated and pixel-averaged, brightness referred to 30 MHz (bottom)",
    fontsize=11, y=1.02,
)
for i, f in enumerate((10.0, 30.0, 50.0)):
    fac = linear_factor(f)
    fac_signed = linear_factor(f, signed=True)
    q, u = sky_qu(f)
    p_avg = (q + 1j * u) * np.exp(2j * lam2(f) * RM) * fac_signed
    q30 = p_avg.real * (30.0 / f) ** BETA_QU
    hp.mollview(np.clip(fac, 1e-4, 1), sub=(2, 3, i + 1), norm="log", min=1e-4, max=1,
                cmap="viridis", title=f"{f:g} MHz: pixel-average factor", cbar=True, notext=True)
    hp.mollview(q30, sub=(2, 3, i + 4), min=-300, max=300, cmap="RdBu_r",
                title=f"{f:g} MHz: pixel-averaged Q", unit="K (at 30 MHz)", cbar=True, notext=True)
    print(f"{f:g} MHz: rms factor {np.sqrt(np.mean(fac**2)):.4f}, median {np.median(fac):.4f}; "
          f"rms Q referred to 30 MHz {np.sqrt(np.mean(q30**2)):.0f} K")
save(fig, "06_depol_factor_maps")


**Figure 6a.** Top: the per-pixel depolarization factor
$|\langle e^{2i\lambda^2\mathrm{RM}}\rangle_\mathrm{pix}|$ under linear RM
interpolation at 10, 30 and 50 MHz, computed analytically from the rms RM
step to each pixel's four edge neighbours (shared log scale, $10^{-4}$ to 1).
Bottom: WMAP K $Q$ rotated with the pixel-centre RM and multiplied by that
factor, with the brightness referred to 30 MHz so the three panels share a
$\pm300$ K scale. This is the exact version of the right panel of Fig. 4d:
the Galactic plane goes to the display floor instead of sitting on the
$\sim 1\%$ random-walk residual of 16384 sub-pixels, and no higher nside is
needed. Sky rms of the factor: 0.003, 0.025, 0.071 at 10, 30, 50 MHz. The
rotated maps are a speckle with that amplitude; what the beam reads from
them is set by the pixel scale, not by the sky.


In [ ]:
# 6b. The ladder along a 1-D cut through the Fig. 4c spot: pixel values are
# the same; only the assumption about RM between them differs.
rng = np.random.default_rng(7)
N_PIX = 5
pix0 = hp.ang2pix(NSIDE, 250.05, -16.18, lonlat=True)
chain = [pix0]
for _ in range(N_PIX // 2):  # walk east and west along the grid
    chain.append(hp.get_all_neighbours(NSIDE, chain[-1])[5])
    chain.insert(0, hp.get_all_neighbours(NSIDE, chain[0])[1])
chain = np.array(chain)
lon, lat = hp.pix2ang(NSIDE, chain, lonlat=True)
centres = (lon - lon[N_PIX // 2]) * np.cos(np.radians(lat))
centres = np.where(centres > 180, centres - 360, centres)
pix_deg = float(np.mean(np.diff(centres)))
rm_c = RM[chain]
x = np.linspace(centres[0] - pix_deg / 2, centres[-1] + pix_deg / 2, 4001)
nn = rm_c[np.clip(np.round(x / pix_deg).astype(int) + N_PIX // 2, 0, N_PIX - 1)]
lin = np.interp(x, centres, rm_c)


def gaussian_subpixel(alpha, dphi1):
    # Zero-mean Gaussian field with D(t) = dphi1^2 (t/1deg)^alpha, conditioned
    # to vanish at the pixel centres so the map values are preserved.
    def cov(a, b):
        d = np.abs(a[:, None] - b[None, :])
        return 0.5 * (dphi1**2 * 20.0**alpha - dphi1**2 * d**alpha)
    kxx, kxc = cov(x, x), cov(x, centres)
    kcc = cov(centres, centres) + 1e-9 * np.eye(N_PIX)
    a = np.linalg.solve(kcc, kxc.T).T
    kpost = kxx - a @ kxc.T
    w, v = np.linalg.eigh((kpost + kpost.T) / 2)
    return (v * np.sqrt(np.clip(w, 0, None))) @ rng.standard_normal(x.size)


cases = [("nearest neighbour\n(the pixel sum)", nn), ("linear interpolation\n(smoothest)", lin)]
for alpha in (2.0, 5 / 3, 1.0):
    cases.append((f"linear + Gaussian sub-pixel\n$\\alpha$={alpha:.3g}, "
                  r"$\Delta\phi_{1^\circ}$=1", lin + gaussian_subpixel(alpha, 1.0)))
F_LAD, ZOOM = 30.0, 0.012
fig, axes = plt.subplots(2, len(cases), figsize=(15, 5.6))
for j, (name, rm) in enumerate(cases):
    ax = axes[0, j]
    ax.plot(x, rm, color="k", lw=1)
    ax.plot(centres, rm_c, "o", color="C3", ms=5)
    for c in np.r_[centres - pix_deg / 2, centres[-1] + pix_deg / 2]:
        ax.axvline(c, color="0.8", lw=0.6)
    ax.set_title(name, fontsize=10)
    ax.set_ylim(rm_c.min() - 3, rm_c.max() + 3)
    ax2 = axes[1, j]
    ax2.plot(x, np.cos(2 * lam2(F_LAD) * rm), color="C0", lw=0.9)
    ax2.set_xlim(-ZOOM, ZOOM)
    ax2.set_ylim(-1.15, 1.15)
    ax2.set_xlabel("offset along cut [deg]")
axes[0, 0].set_ylabel("RM [rad m$^{-2}$]")
axes[1, 0].set_ylabel(rf"$Q/|P_0|=\cos(2\lambda^2 RM)$ at {F_LAD:g} MHz")
fig.suptitle(
    "RM below the map scale under three assumptions (top; red: nside-512 pixel values "
    "along a cut at (l,b)=(250.05,-16.18), grey: pixel edges)\n"
    f"and the 30 MHz fringes they imply inside the central pixel (bottom: the central "
    rf"$\pm${ZOOM}$^\circ$, one fourteenth of the {pix_deg:.2f}$^\circ$ pixel)",
    fontsize=10.5,
)
fig.tight_layout(rect=(0, 0, 1, 0.93))
save(fig, "07_subpixel_ladder")


**Figure 6b.** The same five nside-512 pixel values (red) along a cut
through the Fig. 4c spot, with RM between them taken as nearest neighbour,
linear, or linear plus a Gaussian sub-pixel field with
$\Delta\phi_{1^\circ} = 1$ rad m$^{-2}$ and $\alpha = 2$, 5/3, 1, conditioned
to pass through the pixel values (one realization each). Below, the 30 MHz
fringes $\cos(2\lambda^2\mathrm{RM})$ inside the central pixel, showing one
fourteenth of it. Nearest neighbour has no fringe: that is why the pixel sum
kept every pixel at full amplitude. The $\alpha = 2$ field is as smooth as
the ramp; rougher $\alpha$ scrambles the fringe. A single pixel's average is
not monotonic in roughness (a perfect ramp over many turns sits near a sinc
zero), so the quantitative comparison is the ensemble rms of Fig. 6c.


In [ ]:
# 6b'' context: where the Fig. 6b' patch sits, in two zoom steps.
PATCH_L,PATCH_B,PATCH_DEG=250.0,-15.0,5.0
L0,B0,LAD_ARCMIN=250.049,-16.179,15.0
RM_NORM=Normalize(-150,150)
fig=plt.figure(figsize=(12,4.2))
# left: Mollweide with the 5-degree patch
ax0=fig.add_axes([0.01,0.08,0.58,0.84])
hp.mollview(np.clip(RM,-150,150),fig=fig,hold=True,cmap='RdBu_r',min=-150,max=150,title='Hutschenreuter et al. (2022) RM, nside 512',unit='RM [rad m$^{-2}$]',notext=True)
for d in (PATCH_DEG/2,):
    t=np.linspace(0,2*np.pi,200); lon=PATCH_L+d*np.cos(t)/np.cos(np.radians(PATCH_B)); lat=PATCH_B+d*np.sin(t)
    hp.projplot(lon,lat,lonlat=True,color='k',lw=1.5)
hp.projtext(PATCH_L,PATCH_B-9,'Fig. 4b patch (5$^\\circ$)',lonlat=True,fontsize=10,ha='center',va='top')
# right: 5-degree gnomonic patch with the 15' box
RESO=0.5
gproj=hp.projector.GnomonicProj(rot=(PATCH_L,PATCH_B),xsize=int(PATCH_DEG*60/RESO),reso=RESO)
img=gproj.projmap(RM,lambda x,y,z: hp.vec2pix(NSIDE,x,y,z))
ext=np.degrees(np.array(gproj.get_extent()))
ax1=fig.add_axes([0.62,0.08,0.3,0.84])
ax1.imshow(img,origin='lower',extent=ext,cmap='RdBu_r',norm=RM_NORM,interpolation='nearest')
bx,by=gproj.ang2xy(np.radians(90-B0),np.radians(L0)); bx,by=np.degrees(bx),np.degrees(by)
h=LAD_ARCMIN/60/2
ax1.add_patch(plt.Rectangle((bx-h,by-h),2*h,2*h,fill=False,ec='k',lw=1.5,ls='--'))
ax1.annotate('Fig. 7 patch (15$^\\prime$)',xy=(bx+h,by),xytext=(bx+0.6,by+0.9),fontsize=10,arrowprops=dict(arrowstyle='->',lw=1))
ax1.set_title(f'the 5$^\\circ$ patch at (l, b) = ({PATCH_L:g}$^\\circ$, {PATCH_B:g}$^\\circ$), native pixels',fontsize=10)
ax1.set_xticks([]); ax1.set_yticks([])
cax=fig.add_axes([0.93,0.15,0.012,0.7]); fig.colorbar(plt.cm.ScalarMappable(norm=RM_NORM,cmap='RdBu_r'),cax=cax,label='RM [rad m$^{-2}$]')
save(fig, '07_context')


**Figure 6b′ context.** Where the ladder patch is. Left: the Hutschenreuter
RM map (Fig. 3's scale) with the $5^\circ$ patch of Fig. 4b marked. Right:
that $5^\circ$ patch at native nside-512 pixels, with the $15'$ patch of
Fig. 6b′ dashed. The ladder patch lies in a smooth, low-RM part of the
sky off the plane, chosen in Fig. 4c because its neighbour RM step is close
to the sky median; its regular linear fringes are typical, not favourable.


In [ ]:
# 6b'. The ladder in two dimensions on the Fig. 4c patch: nearest neighbour,
# linear, and linear + Gaussian sub-pixel fields (alpha 5/3 and 1) conditioned
# to pass through the pixel values. Same white noise for both alphas.
from matplotlib.patches import ConnectionPatch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
L0,B0=250.049,-16.179; PATCH_ARCMIN=15.0; RESO_ARCSEC=0.5
NX=int(PATCH_ARCMIN*60/RESO_ARCSEC)  # 1800
F_LAD2 = 30.0
DPHI1 = 1.0
lam2_l = lam2(F_LAD2)
proj=hp.projector.GnomonicProj(rot=(L0,B0), xsize=NX, reso=RESO_ARCSEC/60)
ext=proj.get_extent()
ix,iy=proj.ij2xy(); vec=[v.ravel() for v in proj.xy2vec(ix,iy)]
th,ph=hp.vec2ang(np.stack(vec,axis=1))
pix=hp.vec2pix(NSIDE,*vec).reshape(ix.shape)
rm_nn=RM[pix]
rm_lin=hp.get_interp_val(RM,th,ph).reshape(ix.shape)
# grid coordinates in arcmin (gnomonic x,y are in radians-ish projection units; use degrees from extent)
xg=np.linspace(ext[0],ext[1],NX); yg=np.linspace(ext[2],ext[3],NX)
unit_deg=np.degrees(1.0)  # GnomonicProj xy are tangent-plane units (radians) -> degrees
X,Y=np.meshgrid(xg*unit_deg*60, yg*unit_deg*60)  # arcmin
dx=(X[0,1]-X[0,0])
# pixel centres near the patch
cen=hp.query_disc(NSIDE, hp.ang2vec(L0,B0,lonlat=True), np.radians(PATCH_ARCMIN/60*0.9))
cth,cph=hp.pix2ang(NSIDE,cen); cx,cy=proj.ang2xy(cth,cph); cx=cx*unit_deg*60; cy=cy*unit_deg*60
rng=np.random.default_rng(11)
white=rng.standard_normal((2*NX,2*NX))
kx=np.fft.fftfreq(2*NX,d=dx); KX,KY=np.meshgrid(kx,kx); K=np.hypot(KX,KY); K[0,0]=np.inf
def gaussian_field(alpha):
    amp=K**(-(alpha+2)/2)
    z=np.real(np.fft.ifft2(np.fft.fft2(white)*amp))[:NX,:NX]
    lag=int(round(1.0/dx))  # 1 arcmin
    d_emp=np.mean((z[:,lag:]-z[:,:-lag])**2)
    z*=np.sqrt(DPHI1**2*(1/60)**alpha/d_emp)
    # condition to vanish at pixel centres (kriging with the same structure function)
    def cov(r): return 0.5*(DPHI1**2*(600/60)**alpha - DPHI1**2*(r/60)**alpha)
    Rcc=np.hypot(cx[:,None]-cx[None,:], cy[:,None]-cy[None,:]); Kcc=cov(Rcc)+1e-9*np.eye(cen.size)
    # z at centres by bilinear interpolation
    from scipy.interpolate import RegularGridInterpolator
    zi=RegularGridInterpolator((Y[:,0],X[0,:]), z, bounds_error=False, fill_value=0.0)
    zc=zi(np.stack([cy,cx],axis=1))
    w=np.linalg.solve(Kcc,zc)
    corr=np.zeros_like(z)
    for wc,x0,y0 in zip(w,cx,cy): corr+=wc*cov(np.hypot(X-x0,Y-y0))
    return z-corr
cases=[('nearest neighbour\n(the old pixel sum)',rm_nn),('linear interpolation\n(smoothest)',rm_lin)]
for a in (5/3,1.0):
    g=gaussian_field(a); cases.append((f'linear + Gaussian sub-pixel\n'+r'$\alpha$ = '+f'{a:.3g}', rm_lin+g))
    c=np.hypot(X,Y)<3.0; print(f'alpha {a:.3g}: rms added RM within 3 arcmin of centre {g[c].std():.3f} rad/m^2 -> {2*lam2_l*g[c].std():.0f} rad at 30 MHz')
# pixel outlines
outl=[]
for p in cen:
    b=hp.boundaries(NSIDE,p,step=8); bth,bph=hp.vec2ang(b.T); bx,by=proj.ang2xy(bth,bph); outl.append((bx*unit_deg*60,by*unit_deg*60))
vmin=min(c[1].min() for c in cases[:2])-0.3; vmax=max(c[1].max() for c in cases[:2])+0.3
ZOOM=0.5  # arcmin half-width of the fringe zoom
zc=(np.abs(X[0,:])<=ZOOM); zr=(np.abs(Y[:,0])<=ZOOM)
fig,axes=plt.subplots(2,4,figsize=(15,7.6),constrained_layout=True)
for j,(name,rm) in enumerate(cases):
    ax=axes[0,j]
    im=ax.imshow(rm,origin='lower',extent=(X.min(),X.max(),Y.min(),Y.max()),cmap='viridis',vmin=vmin,vmax=vmax)
    for bx,by in outl: ax.plot(bx,by,color='w',lw=0.7)
    ax.plot(cx,cy,'.',color='w',ms=4)
    ax.add_patch(plt.Rectangle((-ZOOM,-ZOOM),2*ZOOM,2*ZOOM,fill=False,ec='red',lw=1.2,ls='--'))
    ax.set_xlim(X.min(),X.max()); ax.set_ylim(Y.min(),Y.max()); ax.set_title(name,fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
    ax2=axes[1,j]
    q=np.cos(2*lam2_l*rm[np.ix_(zr,zc)])
    ax2.imshow(q,origin='lower',extent=(-ZOOM,ZOOM,-ZOOM,ZOOM),cmap='RdBu_r',vmin=-1,vmax=1)
    ax2.set_xticks([]); ax2.set_yticks([])
    if j==0: ax2.set_ylabel(f'cos(2$\\lambda^2$RM) at {F_LAD2:g} MHz\nin the dashed 1$\\prime$ box')
    for sx in (-1,1):
        fig.add_artist(ConnectionPatch(xyA=(sx*ZOOM,-ZOOM),coordsA=ax.transData,xyB=(sx*ZOOM,ZOOM),coordsB=ax2.transData,color='red',lw=0.8,ls='--'))
    if j==0:
        ax.add_artist(AnchoredSizeBar(ax.transData,1.0,"1$^\\prime$",'lower left',pad=0.4,color='w',frameon=False,size_vertical=0.05))
        ax2.add_artist(AnchoredSizeBar(ax2.transData,10/60,'10$^{\\prime\\prime}$','lower left',pad=0.4,color='k',frameon=False,size_vertical=0.008))
axes[0,0].set_ylabel(f'RM, {PATCH_ARCMIN:g}$\\prime$ patch at ({L0}, {B0})\nwhite: nside-512 pixel edges, centres')
cb=fig.colorbar(im,ax=axes[0,:].tolist(),orientation='vertical',shrink=0.9,pad=0.01); cb.set_label('RM [rad m$^{-2}$]')
fig.suptitle('The same map pixels under each assumption about the RM between them (Gaussian cases: $\\Delta\\phi_{1^\\circ}$ = 1 rad m$^{-2}$; rendered at 0.5$^{\\prime\\prime}$).' + '\n' + 'Top: RM on the Fig. 4c patch. Bottom: the 30 MHz Q fringes inside the central pixel.',fontsize=11)
save(fig, '07_subpixel_ladder_2d')
# slide version: no caption-like title, short row labels
fig.suptitle('')
axes[0,0].set_ylabel('sub-pixel RM [rad m$^{-2}$]')
axes[1,0].set_ylabel(f'implied Q fringe at {F_LAD2:g} MHz\n(dashed 1$^\\prime$ box)')
save(fig, '07_subpixel_ladder_2d_slide')


**Figure 6b′.** The ladder on the sky. Top: RM over the $15'$ patch of
Fig. 4c at $0.5''$ resolution (white: nside-512 pixel boundaries and
centres), with RM between the pixel values taken as nearest neighbour,
linear (healpy's four-neighbour interpolation, kinked where it switches
neighbours), or linear plus a Gaussian sub-pixel field with
$D(\theta) = \Delta\phi_{1^\circ}^2(\theta/1^\circ)^\alpha$,
$\Delta\phi_{1^\circ} = 1$ rad m$^{-2}$, $\alpha = 5/3$ and $1$,
conditioned to vanish at the pixel centres so the map values are kept
(one realization each, same noise for both $\alpha$). An $\alpha = 2$
field is absorbed by the linear term and is not shown. Bottom: the
30 MHz fringes $\cos(2\lambda^2\mathrm{RM})$ inside the dashed $1'$ box at
the central pixel (connectors mark the zoom; scale bars: $1'$ above,
$10''$ below). Nearest neighbour has no fringe, which is why the pixel
sum kept every pixel at full amplitude. The Gaussian fields add 0.04
($\alpha = 5/3$) and 0.14 ($\alpha = 1$) rad m$^{-2}$ rms within $3'$ of
the centre, i.e. 9 and 29 rad of phase at 30 MHz; for $\alpha = 1$ the
phase changes by 2.4 rad between adjacent $0.5''$ grid points, so that
panel is scrambled below the resolution of the picture, as the model says
it should be.


In [ ]:
# 6c. rms pixel-average factor vs frequency for the three levels. For the
# Gaussian family <|mean|^2> = (2 pi / A) int theta exp[-(theta/theta_c)^alpha] dtheta
# with 2 lambda^4 D(theta_c) = 1, valid for theta_c << pixel.
from scipy.special import gamma as gamma_fn

SIGMA_COL = 10.0  # rad/m^2, the pilot's column rms; sets only the saturation
D1_CHORD = 2 * np.sin(np.radians(1) / 2)
f_grid = np.linspace(10, 50, 161)


def ensemble_rms_factor(alpha, dphi1):
    frac = dphi1**2 / (2 * SIGMA_COL**2)
    theta0 = D1_CHORD / (-np.log1p(-frac)) ** (1 / alpha)
    strength = 4 * (SIGMA_COL * lam2(f_grid)) ** 2
    theta_c = theta0 * (-np.log1p(-1 / strength)) ** (1 / alpha)
    pref = np.sqrt(2 * np.pi * gamma_fn(2 / alpha) / alpha)
    return np.minimum(1.0, pref * theta_c / PIX_RAD)


fig, ax = plt.subplots(figsize=(7.2, 4.6))
colors = {2.0: "C0", 5 / 3: "C2", 1.0: "C3"}
styles = {0.3: ":", 1.0: "-", 3.0: "--"}
for alpha in (2.0, 5 / 3, 1.0):
    for dphi1 in (0.3, 1.0, 3.0):
        ax.plot(f_grid, ensemble_rms_factor(alpha, dphi1), color=colors[alpha], ls=styles[dphi1], lw=1.4,
                label=(rf"Gaussian sub-pixel, $\alpha$={alpha:.3g}, $\Delta\phi_{{1^\circ}}$={dphi1:g}"
                       if dphi1 == 1.0 else None))
ax.plot([], [], color="0.5", ls=":", label=r"$\Delta\phi_{1^\circ}$=0.3 rad m$^{-2}$")
ax.plot([], [], color="0.5", ls="--", label=r"$\Delta\phi_{1^\circ}$=3 rad m$^{-2}$")
lin_pts = [np.sqrt(np.mean(linear_factor(f) ** 2)) for f in (10.0, 30.0, 50.0)]
ax.plot([10, 30, 50], lin_pts, "ko", ms=7, label="linear interpolation of the map (sky rms of |sinc| per pixel)")
ax.axhline(1, color="C1", lw=2, label="nearest neighbour (the pixel sum): no depolarization")
ax.set_yscale("log")
ax.set_ylim(1e-4, 2)
ax.set_xlim(10, 50)
ax.set_xlabel("frequency [MHz]")
ax.set_ylabel(r"rms pixel-average factor  $\sqrt{\langle|\langle e^{2i\lambda^2 RM}\rangle_{pix}|^2\rangle}$")
ax.set_title(rf"How much polarized amplitude survives averaging over an nside-512 pixel ({np.degrees(PIX_RAD):.3f}$^\circ$)", fontsize=10)
ax.legend(fontsize=7.5, loc="lower right")
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
save(fig, "08_pixel_factor_vs_freq")


**Figure 6c.** rms pixel-average amplitude factor against frequency for
the three levels. Nearest neighbour (orange) is 1 at every frequency: no
depolarization inside the pixel. Black points: linear interpolation of the
map, the sky rms of the per-pixel $|\mathrm{sinc}|$ of Fig. 6a. Lines: the
Gaussian sub-pixel family, ensemble rms, for $\alpha = 2$ (blue), 5/3
(green), 1 (red) and anchors $\Delta\phi_{1^\circ} = 0.3$ (dotted), 1 (solid),
3 (dashed) rad m$^{-2}$; the map's own rms at $1^\circ$ is 2.4 at
$|b| > 60^\circ$ and 4.4 at $|b| > 30^\circ$. The linear-interpolation sky
lies between the $\alpha = 2$ curves anchored at 1 and 3, because the
neighbour steps of the nside-512 map are steeper than its $1^\circ$ rms
extrapolated smoothly. At fixed anchor, rougher $\alpha$ leaves less; at fixed
$\alpha$ a larger anchor leaves less. Everything the beam measures at strong
rotation inherits this factor squared, times the number of pixels in the beam.


### 6d. Why nearest neighbour could not converge, and why small scales win

The pixel sum is a Riemann sum of $B\,P\,e^{2i\lambda^2\mathrm{RM}}$ over
pixel centres, and a Riemann sum of a phase converges only once the phase
step between samples is below $\pi$. At 130 rad per pixel the samples are
random phases: the sum of $N$ random unit vectors has length $\sqrt N$, so
after normalizing by $N$ the amplitude falls as $1/\sqrt N$ and the power
as $1/N$ (Fig. 5). Resolving the posterior-mean field would need
nside $\sim 3\times10^5$, and would then give the linear-interpolation
answer: a converged integral of a smooth model, not of the sky. The walk in
the complex plane makes both points visible.


In [ ]:
# 6d. The beam-weighted sum as a phasor walk, 30 MHz, cos^2 lobe of 20 deg.
LMAX_SYN=1023
alm=hp.map2alm(RM, lmax=LMAX_SYN, iter=1)
F_WALK = 30.0
lam2_w = lam2(F_WALK)
l0,b0=250.,-15.; vec=hp.ang2vec(l0,b0,lonlat=True); RADIUS=np.radians(20)
def walk(nside, mode):
    rm=hp.alm2map(alm, nside, lmax=LMAX_SYN)  # the same band-limited field at every nside
    pix=hp.query_disc(nside, vec, RADIUS)
    ang=hp.rotator.angdist(hp.pix2vec(nside,pix), vec)
    w=np.cos(np.pi/2*ang/RADIUS)**2 * (4*np.pi/hp.nside2npix(nside))  # lobe x pixel area
    if mode=='nn':
        fac=np.ones(pix.size)
    else:
        nb=hp.get_all_neighbours(nside,pix)[[1,3,5,7]]; ok=nb>=0
        d=np.where(ok, rm[np.where(ok,nb,0)]-rm[pix][None,:], np.nan)
        step=np.sqrt(np.nanmean(d**2,axis=0)); fac=np.abs(np.sinc(2*lam2_w*step/(2*np.pi)))
    # order the steps by angular distance from the centre so the walk is a radial sweep
    o=np.argsort(ang)
    terms=(w*fac*np.exp(2j*lam2_w*rm[pix]))[o]
    return np.cumsum(terms), np.sum(w), pix.size, np.sqrt(np.sum(np.abs(terms)**2))
cases=[('nearest neighbour, nside 512','nn',512),('nearest neighbour, nside 1024','nn',1024),('pixel-averaged (linear), nside 512','lin',512),('pixel-averaged (linear), nside 1024','lin',1024)]
fig,axes=plt.subplots(2,2,figsize=(10.5,9.6))
res={}
walks=[walk(nside,mode)+(name,) for (name,mode,nside) in cases]
for row,(scale_name,idx) in enumerate((('nn',(0,1)),('lin',(2,3)))):
    lim=1.15*max(np.max(np.abs(np.r_[walks[k][0].real,walks[k][0].imag]))/walks[k][1] for k in idx)
    for col,k in enumerate(idx):
        cs,norm,n,rw,name=walks[k]; res[name]=(abs(cs[-1])/norm, rw/norm, n)
        ax=axes[row,col]
        ax.plot(cs.real/norm, cs.imag/norm, lw=0.4, color='C0', alpha=0.9)
        ax.plot([0,cs.real[-1]/norm],[0,cs.imag[-1]/norm],color='C3',lw=2)
        ax.plot(0,0,'k+',ms=8)
        ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_aspect('equal')
        ax.set_title(f'{name}, {n} pixels\n|end point| = {abs(cs[-1])/norm:.1e}, random-walk expectation $\\sqrt{{\\sum|w_p|^2}}$ = {rw/norm:.1e}',fontsize=9.5)
        if row==1: ax.set_xlabel('Re (beam-weighted sum) / unrotated sum')
        if col==0: ax.set_ylabel('Im (beam-weighted sum) / unrotated sum')
fig.suptitle(f'The beam-weighted sum as a walk in the complex plane at {F_WALK:g} MHz: cos$^2$ lobe of 20$^\\circ$ radius at (l,b)=({l0:g},{b0:g}), uniform intrinsic polarization.' + '\n' + 'Hutschenreuter RM synthesized to $\\ell$=1023, so refining nside changes only the sampling. Unrotated, every arrow is aligned and the sum is 1.', fontsize=10.5)
fig.tight_layout(rect=(0,0,1,0.94)); save(fig, '09_phasor_walk')
print(res)


**Figure 6d.** The beam-weighted sum $\sum_p w_p\,f_p\,e^{2i\lambda^2\mathrm{RM}_p}$
at 30 MHz as a walk in the complex plane, for a $\cos^2$ lobe of $20^\circ$
radius at $(l, b) = (250^\circ, -15^\circ)$ with uniform intrinsic
polarization, so only the Faraday factor is summed; pixels are added in
order of distance from the lobe centre. The RM map is synthesized to
$\ell = 1023$ so that refining nside changes only the sampling. Red: the
end point, normalized to the unrotated sum. Top: nearest neighbour
($f_p = 1$) at nside 512 and 1024 on a shared scale; the random-walk
expectation $\sqrt{\sum_p |w_p|^2}$ halves when the pixel count
quadruples, and the end points follow it. Bottom: the pixel-averaged
(linear) sum, $f_p = |\mathrm{sinc}(g_p/2)|$, on its own scale 150 times
smaller; its expectation is the same at both resolutions, and the end
points scatter around it as single realizations do. The unrotated sum is 1
on these axes. Everything the beam measures at strong rotation is this
random walk; its length is set by the number and size of coherent cells,
not by the large-scale polarized sky.


### 6e. The linear-interpolation sky through the real instrument

`scripts/slides_subpixel_spectra.py` takes the pixel-averaged rotated sky
of Fig. 6a at 201 node frequencies across the 30 MHz band (1 kHz apart;
the speckle's frequency coherence is ~11 kHz), degrades it to nside 64
*after* rotation and averaging (a linear low-pass; the beam reads
$\ell \le 48$), and sends it through the production arm: the as-built
four-port response, the loaded covariance frozen at the native channel,
parent/zoom channelization and the zenith-calibrated pseudo-Stokes. A
uniform 1 K unpolarized sky through the same pipeline is the calibration,
so the axes are kelvin of beam-weighted brightness. Run
`.venv/bin/python scripts/slides_subpixel_spectra.py compute --center 30`
and `--center 50` first (about 3 min each); the cell below only plots
from the cache.


In [ ]:
import importlib, sys
sys.path.insert(0, str(Path("../scripts").resolve()))
spx = importlib.import_module("slides_subpixel_spectra")
from IPython.display import Image, display
for center in (30.0, 50.0):
    spx.set_center(center)
    if (spx.out_dir() / "spectra.npz").exists():
        spx.plot(spx.out_dir())
        display(Image(filename=str(spx.FIG_DIR / f"10_subpixel_spectra_{center:g}MHz.png")))
    else:
        print(f"run: .venv/bin/python scripts/slides_subpixel_spectra.py compute --center {center:g}")


**Figure 6e.** Top: the full pseudo-Stokes $Q$ across the 64 zoom bins of
the central 25 kHz parent channel at 30 MHz, at one sidereal phase: the
I-only sky (leakage $I \to Q$ alone), leakage plus the unrotated WMAP
polarization (no Faraday rotation, a smooth offset of a few hundred K), and
leakage plus the Faraday-rotated polarization, which lies on top of the
leakage curve. Leakage does not depend on the polarized sky; rotation
collapses the polarized term from hundreds of kelvin to the speckle below.
Middle and bottom: pseudo-Stokes $Q$ and $U$ of the Faraday polarized signal
alone, for four sidereal phases. Red shading marks the aliased edge zoom
bins, whose polyphase response folds more than 5% of its weight (30 to 44%
for the outermost bins) from the far end of the same parent channel; the
seams between parents in the three-parent variant
(`--span 3`) are that folding, not sky structure.
Grey bands: $\pm1$ rms of the ensemble prediction from the column-screen
covariance run (`configs/paper/covariance-column.json`, $\alpha = 2$,
anchors 1 and 3 rad m$^{-2}$ at $1^\circ$), scaled from its 1 K intrinsic
normalization to WMAP K's pixel rms at 30 MHz ($1.26\times10^4$ K, all
scales, WMAP noise included, which is what the realization contains). The
realization's zoom rms is 0.02 to 0.17 K and sits inside the anchor-1 band
and at the edge of the anchor-3 band, where Fig. 6c placed the linear sky.
The beam-weighted $I$ is $1.6$ to $3.5\times10^4$ K and the I-only leakage
$Q$ is 500 to 4400 K, so the speckle is $10^{-6}$ to $10^{-5}$ of $I$
against a leakage of $0.03$ to $0.15$ of $I$. The ripple's 5 to 10 kHz
coherence inside the parent channel is the frequency signature; the parent
channel averages over it. The 50 MHz companion (second figure) has a zoom
rms of 0.003 to 0.04 K against $I$ of 4.3 to $9.1\times10^3$ K, the same
$10^{-6}$ to $10^{-5}$ of $I$; its speckle coherence (~49 kHz) is wider than
the parent channel, so the ripple there is a slow drift across the zoom
band rather than a within-channel wiggle.


### 6f. Why the edge zoom bins fold

The zoom bins are a 64-point transform of one parent channel's time stream. A bin near the parent edge therefore has an image at the mirror frequency beyond the far edge, and the parent's own passband (left) extends past its nominal $\pm12.5$ kHz, so that image is not suppressed. This is the readout, not the sky; it is why the outer bins are shaded in Fig. 6e.


In [ ]:
# 6f. The spectrometer response behind the aliased edge zoom bins.
from lusee_faraday import channelization as chan
off=np.arange(-60e3,60e3,chan.FINE_STEP_HZ)
wp=chan.parent_weights(off)
wz=chan.zoom_weights(off)
centres=chan.zoom_bin_offsets_hz()
fig,axes=plt.subplots(1,2,figsize=(13,6.0))
ax=axes[0]
ax.semilogy(off/1e3, wp/wp.max(), color='k', lw=1.5)
for e in (-12.5,12.5): ax.axvline(e,color='0.5',ls=':',lw=0.8)
ax.set_xlim(-60,60); ax.set_ylim(1e-6,2); ax.set_xlabel('offset from the parent channel centre [kHz]'); ax.set_ylabel('response, normalized to peak')
ax.set_title('one 25 kHz parent channel (polyphase filter bank response)',fontsize=10); ax.grid(alpha=0.3,which='both')
ax=axes[1]
inside=np.abs(off)<=12.5e3
for k,c in zip((0,16,24,30,32),('C0','C2','C1','C3','C4')):
    w=np.abs(wz[:,k]); w=w/w.max()
    folded=np.sum(np.abs(wz[~inside,k]))/np.sum(np.abs(wz[:,k]))
    ax.semilogy(off/1e3, w, color=c, lw=1.3, label=f'bin at {centres[k]/1e3:+.2f} kHz: {100*folded:.0f} % outside the parent')
for e in (-12.5,12.5): ax.axvline(e,color='0.5',ls=':',lw=0.8)
ax.axvspan(-30,-12.5,color='C3',alpha=0.06); ax.axvspan(12.5,30,color='C3',alpha=0.06)
ax.set_xlim(-25,25); ax.set_ylim(1e-4,2); ax.set_xlabel('offset from the parent channel centre [kHz]')
ax.set_title('zoom bins of that parent (25 kHz / 64 = 390.6 Hz each)',fontsize=10); ax.grid(alpha=0.3,which='both')
h,l=ax.get_legend_handles_labels(); fig.legend(h,l,fontsize=9,loc='lower center',ncol=3,bbox_to_anchor=(0.5,0.0))
fig.suptitle('LuSEE spectrometer response (luseepy SpectrometerResponse); dotted lines: the nominal parent edges at $\\pm$12.5 kHz.' + '\n' + 'An edge zoom bin responds almost as strongly at its mirror frequency beyond the far edge as at its own centre: the folded, aliased bins.',fontsize=10)
fig.tight_layout(rect=(0,0.12,1,0.9)); save(fig, '11_spectrometer_response')


**Figure 6f.** The LuSEE spectrometer response from luseepy's `SpectrometerResponse`. Left: one 25 kHz parent channel; its passband is flat to about $\pm18$ kHz, wider than the nominal channel, so neighbouring parents overlap. Right: five zoom bins of that parent on a $\pm25$ kHz axis. The central and mid-channel bins respond only inside the parent. The bins near the edge respond at their own centre and, almost as strongly, at the mirror frequency beyond the far edge (the folded image), so 5 %, 32 % and 45 % of their response lies outside the parent for the bins at +9.4, +11.7 and $-12.5$ kHz. Those are the bins shaded in Fig. 6e, and the seams between parents in its three-parent variant are this folding.
